# Bulk Archive Mining with seisfetch

This notebook demonstrates quakescope-style data mining:
fetching waveforms from hundreds of station-day combinations
across EarthScope, SCEDC, and NCEDC in parallel.

The workflow:
1. Build a request list (from code, CSV, or station queries)
2. Parallel bulk fetch → numpy arrays
3. Monitor progress
4. Save results to disk (npz or zarr)

**No ObsPy anywhere in this pipeline.**

In [ ]:
from pathlib import Path

import numpy as np

from seisfetch import (
    SeisfetchClient,
    requests_from_csv,
    requests_from_list,
    route_network,
)

## 1. Build request lists

Requests can be tuples, dicts, `BulkRequest` objects, or read from CSV.

In [ ]:
# From tuples: (network, station, location, channel, starttime, endtime)
requests_tuples = [
    ("IU", "ANMO", "00", "BHZ", "2024-01-15", "2024-01-15T01:00:00"),
    ("IU", "ANMO", "00", "BHN", "2024-01-15", "2024-01-15T01:00:00"),
    ("IU", "ANMO", "00", "BHE", "2024-01-15", "2024-01-15T01:00:00"),
    ("IU", "CCM", "00", "BHZ", "2024-01-15", "2024-01-15T01:00:00"),
    ("IU", "WVT", "00", "BHZ", "2024-01-15", "2024-01-15T01:00:00"),
]

# The helper normalizes any input format
reqs = requests_from_list(requests_tuples)
print(f"Built {len(reqs)} requests")
for r in reqs:
    dc = route_network(r.network)
    print(f"  {r.tag} [{r.starttime}] → {dc}")

### Cross-datacenter requests

A single bulk job can span all three S3 buckets — routing is automatic.

In [ ]:
cross_dc_requests = [
    # EarthScope
    ("IU", "ANMO", "00", "BHZ", "2024-06-01", "2024-06-01T01:00:00"),
    ("UW", "MBW", "00", "HHZ", "2024-06-01", "2024-06-01T01:00:00"),
    # SCEDC
    ("CI", "SDD", "", "BHZ", "2024-06-01", "2024-06-01T01:00:00"),
    ("CI", "DJJ", "", "BHZ", "2024-06-01", "2024-06-01T01:00:00"),
    # NCEDC
    ("BK", "BRK", "00", "BHZ", "2024-06-01", "2024-06-01T01:00:00"),
    ("BK", "CMB", "00", "BHZ", "2024-06-01", "2024-06-01T01:00:00"),
]

for net, sta, *_ in cross_dc_requests:
    print(f"  {net}.{sta:5s} → {route_network(net)}")

### From CSV file

For large mining campaigns, prepare a CSV:

```csv
# network,station,location,channel,starttime,endtime
IU,ANMO,00,BHZ,2024-01-15,2024-01-16
CI,SDD,,BHZ,2024-06-01,2024-06-02
BK,BRK,00,BHZ,2024-06-01,2024-06-02
```

In [ ]:
# Write a sample CSV
csv_path = Path("sample_requests.csv")
csv_path.write_text(
    "# network,station,location,channel,starttime,endtime\n"
    "IU,ANMO,00,BHZ,2024-01-15,2024-01-15T01:00:00\n"
    "IU,CCM,00,BHZ,2024-01-15,2024-01-15T01:00:00\n"
)

reqs_csv = requests_from_csv(csv_path)
print(f"Loaded {len(reqs_csv)} requests from CSV")

## 2. Bulk fetch → numpy

`get_numpy_bulk()` downloads and decodes all requests in parallel.
Each result has `.bundle` (a `TraceBundle` with numpy arrays).

In [ ]:
client = SeisfetchClient(backend="s3_open")


# Progress callback — shows live status
def on_progress(completed, total, result):
    pct = completed / total * 100
    if result.success:
        npts = sum(t.npts for t in result.bundle.traces) if result.bundle else 0
        print(
            f"  [{pct:5.1f}%] {result.request.tag}: "
            f"{result.nbytes:>10,} B, {result.elapsed_s:.2f}s, "
            f"{result.throughput_mbps:.0f} Mbps, {npts:,} samples"
        )
    else:
        print(f"  [{pct:5.1f}%] {result.request.tag}: FAILED — {result.error}")


print("Fetching 5 station-channels in parallel...")
summary = client.get_numpy_bulk(
    requests_tuples,
    max_workers=8,
    progress=on_progress,
)
print(f"\n{summary}")

## 3. Inspect results

In [ ]:
print(f"Succeeded: {summary.succeeded}/{summary.total}")
print(f"Total data: {summary.total_bytes / 1e6:.1f} MB")
print()

for result in summary.successful_results:
    arrays = result.bundle.to_dict()
    for nslc, data in arrays.items():
        print(
            f"  {nslc}: {data.shape} {data.dtype}  "
            f"(min={data.min()}, max={data.max()})"
        )

# Handle failures
for result in summary.failed_results:
    print(f"  FAILED: {result.request.tag} — {result.error}")

## 4. Save results

### Option A: Save as compressed .npz files

In [ ]:
output_dir = Path("bulk_output")
output_dir.mkdir(exist_ok=True)

for result in summary.successful_results:
    arrays = result.bundle.to_dict()
    outfile = output_dir / f"{result.request.tag}.npz"
    np.savez_compressed(str(outfile), **arrays)

print(f"Saved {len(summary.successful_results)} files to {output_dir}/")
for f in sorted(output_dir.glob("*.npz")):
    print(f"  {f.name}  ({f.stat().st_size / 1e3:.1f} KB)")

### Option B: Save as zarr store

In [ ]:
from seisfetch import bundle_to_xarray, to_zarr

# Combine all successful bundles into one xarray Dataset
from seisfetch.convert import TraceBundle

all_traces = []
for result in summary.successful_results:
    all_traces.extend(result.bundle.traces)

combined = TraceBundle(all_traces)
ds = bundle_to_xarray(combined)
print(ds)

# Write to zarr
to_zarr(ds, "bulk_output.zarr")
print("\nSaved to bulk_output.zarr")

## 5. Scaling up

For real mining campaigns (thousands of station-days), tune `max_workers`
to match your bandwidth. From within AWS us-east-2:

```python
# 1000 station-day requests, 32 parallel downloads
summary = client.get_numpy_bulk(requests, max_workers=32)
```

Typical throughput from us-east-2:
- Single S3 object: 500–1000 Mbps
- 32 parallel: 5–10 Gbps aggregate (limited by EC2 network)
- Parse overhead: negligible (pymseed decodes at 30+ Gbps)

### CLI alternative

```bash
seisfetch bulk requests.csv -o output/ -f npz -w 32
```

## Cleanup

In [ ]:
import shutil

csv_path.unlink(missing_ok=True)
shutil.rmtree("bulk_output", ignore_errors=True)
shutil.rmtree("bulk_output.zarr", ignore_errors=True)
print("Cleaned up.")